In [ ]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai faiss-cpu beautifulsoup4


In [ ]:
# ==========================================
# 1. INSTALACIÓN DE LIBRERÍAS
# ==========================================
!pip install -qU langchain-core langchain-community langchain-google-genai faiss-cpu beautifulsoup4 sentence-transformers langchain-huggingface

import os
import getpass
import warnings

warnings.filterwarnings("ignore")
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

# LA SOLUCIÓN: Embeddings locales que no dependen de la API de Google
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("\n" + "="*50)
print("🎓 SISTEMA RAG HÍBRIDO (Local Embeddings + Gemini LLM)")
print("="*50)

# ==========================================
# 2. CONTEXTO INTERACTIVO
# ==========================================
if "GEMINI_API_KEY" not in os.environ:
    print("\n🔑 Ingresa tu Gemini API Key:")
    os.environ["GEMINI_API_KEY"] = getpass.getpass()

print("\n🌐 ¿Qué página web quieres que la IA lea?")
print("   (Presiona Enter para usar la Malla de Ingeniería UNAL)")
url_input = input("> ").strip()
url_final = url_input if url_input else "https://ingenieria.bogota.unal.edu.co/es/programas-academicos/ingenieria-industrial/"

# ==========================================
# 3. EXTRACCIÓN Y VECTORIZACIÓN (100% LOCAL)
# ==========================================
print(f"\n⏳ Leyendo la web: {url_final}...")
try:
    documentos_web = WebBaseLoader(url_final).load()
    print("✅ Web leída con éxito.")
except Exception as e:
    print(f"🚨 Error de lectura web: {e}")
    raise SystemExit

print("🧩 Descargando modelo de Embeddings local y vectorizando en FAISS...")
# Usamos un modelo súper ligero y eficiente de HuggingFace (all-MiniLM-L6-v2)
fragmentos = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(documentos_web)
embeddings_locales = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(fragmentos, embeddings_locales)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 4. LA CADENA LCEL (El cerebro Gemini)
# ==========================================
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

template = """Eres un asistente académico. Usa ÚNICAMENTE el siguiente contexto para responder. Si no lo sabes, di que no tienes esa información en la base de datos.
Contexto recuperado:
{context}

Pregunta del usuario: {question}"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("\n🤖 CEREBRO HÍBRIDO LISTO. ¡Puedes preguntar!")
print("="*50)

# ==========================================
# 5. CHAT CON RAYOS X
# ==========================================
while True:
    pregunta = input("\n💬 Tu pregunta (o 'salir'):\n> ")
    if pregunta.lower() in ['salir', 'exit', 'quit']: break

    print("\n🔍 [Buscando vectores localmente...]")

    docs_recuperados = retriever.invoke(pregunta)
    print("📑 FRAGMENTOS USADOS:")
    for i, doc in enumerate(docs_recuperados):
        print(f"   [{i+1}] {doc.page_content[:150].replace(chr(10), ' ')}...")

    respuesta = rag_chain.invoke(pregunta)
    print(f"\n💡 GEMINI RESPONDE:\n{respuesta}")
    print("-" * 50)


🎓 SISTEMA RAG HÍBRIDO (Local Embeddings + Gemini LLM)

🌐 ¿Qué página web quieres que la IA lea?
   (Presiona Enter para usar la Malla de Ingeniería UNAL)
> 

⏳ Leyendo la web: https://ingenieria.bogota.unal.edu.co/es/programas-academicos/ingenieria-industrial/...
✅ Web leída con éxito.
🧩 Descargando modelo de Embeddings local y vectorizando en FAISS...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🤖 CEREBRO HÍBRIDO LISTO. ¡Puedes preguntar!

💬 Tu pregunta (o 'salir'):
> cual es el objetivo?

🔍 [Buscando vectores localmente...]
📑 FRAGMENTOS USADOS:
   [1] Componente de Libre Elección Este componente permite al estudiante aproximarse, contextualizar y/o profundizar temas de su profesión o disciplina y ap...
   [2] Generar nuevas industrias y propiciar el mejoramiento de las existentes y, como consecuencia, promover el desarrollo social y económico del país y de ...
   [3] La estructura curricular del plan se articula en ciertos campos de formación que tienen la misión de contribuir al proceso integral de preparación de ...

💡 GEMINI RESPONDE:
El objetivo del Componente de Libre Elección es acercar a los estudiantes a las tareas de investigación, extensión, emprendimiento y toma de conciencia de las implicaciones sociales de la generación de conocimiento.
--------------------------------------------------

💬 Tu pregunta (o 'salir'):
> completa el texto de "Generar nuevas industria

KeyboardInterrupt: Interrupted by user


💬 Tu pregunta (o 'salir'):
> salir
